<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Unit of Analysis + Time Window

One row represents one keyword article webpage and its associated SEO performance metrics for a single monthly snapshot.

For this assignment, I use the month **2026-03** as the analysis window because it is a mid-panel month that avoids using the final month as training data. This allows the data to be used for feature engineering and validation without introducing future information.

In [13]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

login(userdata.get("HF_TOKEN"))

performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train"
)

print("Dataset loaded successfully!")
print("Rows:", len(performance))
print("Columns:", performance.column_names)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset loaded successfully!
Rows: 78835655
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Fields

### Features
- Average Position
- CTR
- Search Volume
- Engagement Rate
- Trend Direction

These features are available before making the prioritisation decision and can be used to estimate which webpages deserve review.

### Label / Proxy
- SEO Review Priority Score (proxy)

The dataset does not contain a true "review priority" label, so I define a proxy score based on search performance signals to represent review priority.

### Context
- Month
- Content Type (Keyword Article)

These fields provide filtering and context but are not used directly as predictive features.

### Excluded
- URL

Reason: The URL is an identifier rather than a predictive feature. Including it could cause the model to memorise specific pages instead of learning general patterns.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# Filter only March 2026 records

march_data = performance.filter(
    lambda row: row["report_date"].strftime("%Y-%m") == "2026-03"
)

print("Rows in March 2026:", len(march_data))

Rows in March 2026: 9841378


In [17]:
sample = march_data.select(range(min(1000, len(march_data))))

keys = set()

for row in sample:
    keys.add(
        (
            row["report_date"],
            row["content_hash_id"]
        )
    )

print("Rows checked:", len(sample))
print("Unique (report_date, content_hash_id):", len(keys))

Rows checked: 1000
Unique (report_date, content_hash_id): 1000


#Query 1 Result:

This query verifies that the sampled (report_date, content_hash_id) combinations are unique, supporting the stated unit of analysis of one content item per reporting date.

In [18]:
# Query 2 - Row count and date span for March 2026

sample = march_data.select(range(min(5000, len(march_data))))

dates = sample["report_date"]

print("March 2026 Rows:", len(march_data))
print("Sample Rows Checked:", len(sample))
print("Earliest Date in Sample:", min(dates))
print("Latest Date in Sample:", max(dates))

March 2026 Rows: 9841378
Sample Rows Checked: 5000
Earliest Date in Sample: 2026-03-01
Latest Date in Sample: 2026-03-01


#Query 2 Result:
This query verifies the number of records for the selected March 2026 analysis window and checks the reporting date range using a representative sample of the filtered data.

In [19]:
available = march_data.filter(
    lambda row: row["gsc_data_available"] is True
)

print("Rows with GSC data available:", len(available))

Rows with GSC data available: 3611061


#Query 3 Result:

This query verifies the availability of Google Search Console data using the IS TRUE condition and reports how many rows remain after filtering.

#Data Limits

The warehouse dataset cannot directly determine the true business value or quality of a webpage. It only contains search and analytics performance metrics rather than human quality assessments.

The history available for different clients is uneven because some clients joined the platform later than others. In addition, some early records contain Search Console data without Google Analytics data, so missing analytics values should not be interpreted as zero performance.

This analysis should therefore be treated as decision-support rather than a complete measure of SEO success.

Another limitation is that this analysis relies on a proxy label rather than a true business-defined SEO review priority.

In [20]:
print("No code required for this section.")

No code required for this section.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.